# Food Image Recognition — Final-Exam Experiments (Resumable)

**Course:** Topics on Intelligent Systems  |  **Instructor:** Prof. Shin, Dong Il  
**Student:** Saad Muhammad (26160024)  |  **Lab:** SDC, Sejong University

## ⚡ How resumability works

1. **All outputs are saved to your Google Drive** (`MyDrive/food-recognition-experiments/`). Drive survives Colab runtime disconnects.
2. **Every training cell checks first** if that model is already fully trained. If yes → skips. If no → trains it.
3. **Every evaluation cell checks first** if that model is already evaluated. If yes → skips.

## 😢 If your runtime disconnects mid-training

1. Reconnect runtime (Runtime → Reconnect).
2. Re-run the **Setup** section (cells 1–3).
3. Then **Run all cells from the top**. Completed models will skip. Training resumes from the model that was interrupted.

**Note:** If a model was *partially* trained (e.g., 8 of 15 epochs done), that model restarts from epoch 1 — only fully completed models are skipped. Worst case loss = ~30 min of training.

---
## 🔧 Setup (re-run these 3 cells every time runtime restarts)

In [ ]:
# Cell 1 — Install dependencies
!pip install -q timm tqdm scikit-learn seaborn

In [ ]:
# Cell 2 — Mount Google Drive and link persistent folders
import os, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# Persistent project folder on Drive
DRIVE_PROJECT = '/content/drive/MyDrive/food-recognition-experiments'
os.makedirs(f'{DRIVE_PROJECT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/outputs', exist_ok=True)

# Symlink so your code writes to Drive transparently
for name in ['data', 'outputs']:
    local = Path(name)
    if local.exists() or local.is_symlink():
        local.unlink() if local.is_symlink() else __import__('shutil').rmtree(local)
    local.symlink_to(f'{DRIVE_PROJECT}/{name}')

print('✓ Drive mounted')
print(f'✓ data/    → {DRIVE_PROJECT}/data')
print(f'✓ outputs/ → {DRIVE_PROJECT}/outputs')

# Quick sanity check on previous progress
ckpt_dir = Path('outputs/checkpoints')
log_dir  = Path('outputs/logs')
if ckpt_dir.exists():
    done_ckpts = sorted(p.stem for p in ckpt_dir.glob('*_best.pt'))
    done_evals = sorted(p.stem.replace('_eval','') for p in log_dir.glob('*_eval.json'))
    print(f'\n📦 Previous progress on Drive:')
    print(f'  trained:   {[d.replace("_best","") for d in done_ckpts] or "none"}')
    print(f'  evaluated: {done_evals or "none"}')
else:
    print('\n📦 No previous progress — fresh run.')

sys.path.insert(0, './src')

In [ ]:
# Cell 3 — Config + helpers (defines is_trained() and is_evaluated() guards)
import json, os
from pathlib import Path
import torch

CONFIG = {
    'num_classes': 20,      # subset of Food-101
    'epochs':      15,
    'batch_size':  64,
    'img_size':    224,
    'lr':          1e-4,
    'weight_decay':1e-4,
    'num_workers': 2,
    'seed':        42,
    'data_root':   './data',
    'out_root':    './outputs',
}

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

def is_trained(model_name, target_epochs=None):
    """True iff the model has completed `target_epochs` epochs of training."""
    target_epochs = target_epochs or CONFIG['epochs']
    log_path = Path(f'outputs/logs/{model_name}_log.json')
    if not log_path.exists():
        return False
    with open(log_path) as f:
        history = json.load(f)
    return len(history) >= target_epochs

def is_evaluated(model_name):
    """True iff the model has been evaluated and the eval JSON exists."""
    return Path(f'outputs/logs/{model_name}_eval.json').exists()

def train_if_needed(model_name):
    if is_trained(model_name):
        print(f'✓ {model_name} already trained ({CONFIG["epochs"]} epochs) — skipping')
        return
    print(f'▶ Training {model_name}...')
    !python src/train.py --model {model_name} --epochs {CONFIG['epochs']} --num_classes {CONFIG['num_classes']} --batch_size {CONFIG['batch_size']}

def evaluate_if_needed(model_name):
    if is_evaluated(model_name):
        print(f'✓ {model_name} already evaluated — skipping')
        return
    print(f'▶ Evaluating {model_name}...')
    !python src/evaluate.py --model {model_name} --num_classes {CONFIG['num_classes']}

print('\n✓ Helpers ready: train_if_needed(), evaluate_if_needed()')

---
## 📂 Dataset preview (downloads Food-101 to Drive on first run — ~5 GB)

In [ ]:
from dataset import get_food101_loaders
train_loader, test_loader, info = get_food101_loaders(
    root=CONFIG['data_root'], img_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'], num_classes=CONFIG['num_classes'],
    num_workers=CONFIG['num_workers'], seed=CONFIG['seed']
)
print('Classes used:', info['kept_class_names'])
print(f'Train: {info["num_train_samples"]}  |  Test: {info["num_test_samples"]}')

import matplotlib.pyplot as plt
import numpy as np
x, y = next(iter(train_loader))
mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, lbl in zip(axes.flat, x[:8], y[:8]):
    img = img.permute(1, 2, 0).numpy() * std + mean
    ax.imshow(np.clip(img, 0, 1)); ax.set_title(info['kept_class_names'][lbl]); ax.axis('off')
plt.tight_layout(); plt.show()

---
## 🧮 Model summary (no training — just shows parameter counts)

In [ ]:
from models import MODEL_BUILDERS, build_model, count_parameters
print(f'{"model":18s}  {"params (M)":>10s}  status')
print('-' * 50)
for name in MODEL_BUILDERS:
    m = build_model(name, num_classes=CONFIG['num_classes'], pretrained=False)
    status = '✓ done' if is_trained(name) else 'pending'
    print(f'{name:18s}  {count_parameters(m)/1e6:10.2f}  {status}')
    del m

---
## 🏋️ Training — 8 models, each cell auto-skips if done

Each cell uses `train_if_needed(...)` which:
- Checks if `outputs/logs/<model>_log.json` has all 15 epochs → if yes, skips.
- Otherwise launches `src/train.py`, which saves checkpoint + log after every epoch to Drive.

Approximate per-model wall-clock on Colab T4, 20 classes, 15 epochs:

| Model | Time |
|---|---|
| alexnet | 15–20 min |
| mresnet50, v1–v5 | 25–35 min each |
| vit | 35–45 min |

**Total ≈ 4–5 hours.** Free-tier Colab disconnects after ~90 min idle; just reconnect and re-run from the top — skip logic handles the rest.

In [ ]:
# Baseline 1 — AlexNet (Paper 1)
train_if_needed('alexnet')

In [ ]:
# Baseline 2 — MResNet-50 (Paper 2)
train_if_needed('mresnet50')

In [ ]:
# Baseline 3 — NoisyViT (Paper 3)
train_if_needed('vit')

In [ ]:
# Ablation V1 — Vanilla ResNet-50
train_if_needed('v1_resnet50')

In [ ]:
# Ablation V2 — + Swish activations
train_if_needed('v2_swish')

In [ ]:
# Ablation V3 — + Transformer head
train_if_needed('v3_transformer')

In [ ]:
# Ablation V4 — + Fixed noise (NoisyViT-style)
train_if_needed('v4_fixed_noise')

In [ ]:
# Ablation V5 — PROPOSED + Adaptive Noise Injection ⭐
train_if_needed('v5_proposed')

---
## 📊 Evaluation — 7 metrics per model, auto-skips if done

In [ ]:
for m in ['alexnet','mresnet50','vit','v1_resnet50','v2_swish','v3_transformer','v4_fixed_noise','v5_proposed']:
    evaluate_if_needed(m)

---
## 🎨 Generate all 10 figures + 2 CSV tables

In [ ]:
!python src/plot_results.py

In [ ]:
from IPython.display import Image, display
import glob
for png in sorted(glob.glob('outputs/figures/*.png')):
    print(png); display(Image(png))

---
## 📋 Final tables

In [ ]:
import pandas as pd
print('=== Summary table (all 8 models, 7 metrics) ===')
display(pd.read_csv('outputs/figures/summary_table_all.csv'))
print('\n=== Ablation table (V1–V5 with Δ columns) ===')
display(pd.read_csv('outputs/figures/ablation_table.csv'))

---
## 💾 Download results

Everything is already on Drive at `MyDrive/food-recognition-experiments/outputs/`. This cell also makes a zip for quick download.

In [ ]:
!cd /content && zip -rq outputs.zip outputs/figures outputs/logs
from google.colab import files
files.download('/content/outputs.zip')